<a href="https://colab.research.google.com/github/joselmuniz0/gestion-con-python/blob/main/Copia_de_video_analisis_aut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS
import cv2
import os
from google.colab import drive
from google.colab.patches import cv2_imshow # Para visualizar imágenes en Colab

# 2. CONEXIÓN A GOOGLE DRIVE
print("Conectando con Google Drive...")
drive.mount('/content/drive')

# 3. CONFIGURACIÓN DE RUTAS Y CREACIÓN DE CARPETAS
# Define la ruta principal en tu unidad
ruta_raiz = '/content/drive/MyDrive/Analisis_Deportivo'
carpetas = ['Videos_Crudos', 'Resultados', 'Configuracion']

print("\nVerificando estructura de carpetas...")
for carpeta in carpetas:
    ruta_completa = os.path.join(ruta_raiz, carpeta)
    if not os.path.exists(ruta_completa):
        os.makedirs(ruta_completa)
        print(f"✅ Carpeta creada: {carpeta}")
    else:
        print(f"✔️ Carpeta ya existente: {carpeta}")

# 4. CONFIGURACIÓN DEL VIDEO A ANALIZAR
# @title Configuración del Análisis
# @markdown Escribe el nombre del archivo de video que subiste a 'Videos_Crudos' (ej: partido_test.mp4)
nombre_video = "Hue vsArs - 10 mayo 26.mp4" # @param {type:"string"}

# Construct the video path based on its actual location (root of MyDrive)
ruta_video = os.path.join('/content/drive/MyDrive/', nombre_video)

# 5. EXTRACCIÓN DEL FRAME DE CALIBRACIÓN
if os.path.exists(ruta_video):
    cap = cv2.VideoCapture(ruta_video)

    # Saltamos al segundo 5 para evitar pantallas en negro al inicio
    cap.set(cv2.CAP_PROP_POS_MSEC, 150000)

    success, frame = cap.read()

    if success:
        # Guardamos el frame para usarlo en la calibración
        ruta_frame = os.path.join(ruta_raiz, 'Configuracion', 'frame_calibracion.jpg')
        cv2.imwrite(ruta_frame, frame)

        print("\n¡Éxito! Frame de calibración extraído.")
        print(f"Guardado en: {ruta_frame}")

        # Mostramos una versión pequeña para confirmar
        ancho_muestra = 600
        alto_muestra = int(frame.shape[0] * (ancho_muestra / frame.shape[1]))
        frame_pequeno = cv2.resize(frame, (ancho_muestra, alto_muestra))

        print("\nVista previa del video:")
        cv2_imshow(frame_pequeno)
    else:
        print("❌ Error: No se pudo leer el video. Verifica el formato.")

    cap.release()
else:
    print(f"❌ Error: No se encontró el video '{nombre_video}' en la ruta especificada: {ruta_video}")
    print("Por favor, verifica el nombre y la ubicación del archivo en tu Google Drive.")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import math

# Load the player crop images
frames_jugadores = []
num_samples = len(muestras_colores) # Use the number of detected color samples

for i in range(num_samples):
    sample_image_path = f'/content/muestra_{i}.jpg'
    if os.path.exists(sample_image_path):
        img_sample = cv2.imread(sample_image_path)
        if img_sample is not None:
            frames_jugadores.append(cv2.cvtColor(img_sample, cv2.COLOR_BGR2RGB))
        else:
            print(f"❌ Error: No se pudo cargar la imagen de muestra: {sample_image_path}")
    else:
        print(f"❌ Error: Archivo de muestra no encontrado: {sample_image_path}")

n = len(frames_jugadores) # Actual number of loaded frames
cols = 5
rows = math.ceil(n / cols)

plt.figure(figsize=(cols * 3, rows * 3))

print(f"Mostrando {n} muestras de jugadores en una cuadrícula de {rows}x{cols}:")

for i, frame_player in enumerate(frames_jugadores):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(frame_player)
    plt.title(f"Jugador {i} (HSV: {muestras_colores[i].round(1)})", fontsize=8)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
from google.colab import output
from google.colab.patches import cv2_imshow
from IPython.display import display, HTML, Javascript
import json

# Cargamos el frame guardado anteriormente
ruta_frame = '/content/drive/MyDrive/Analisis_Deportivo/Configuracion/frame_calibracion.jpg' # Corrected path
img = cv2.imread(ruta_frame)
alto, ancho, _ = img.shape

# Función JavaScript para capturar clics
canvas_html = """
<canvas id="canvas" width="{w}" height="{h}" style="border:1px solid #000; cursor: crosshair;"></canvas>
<br><button onclick="sendPoints()">Confirmar Puntos</button>
<script>
    var canvas = document.getElementById('canvas');
    var ctx = canvas.getContext('2d');
    var img = new Image();
    var points = [];

    img.src = "data:image/jpeg;base64,{data}";
    img.onload = function() {{ ctx.drawImage(img, 0, 0); }};

    canvas.onclick = function(e) {{
        var rect = canvas.getBoundingClientRect();
        var x = e.clientX - rect.left;
        var y = e.clientY - rect.top;
        points.push([x, y]);
        ctx.fillStyle = "red";
        ctx.beginPath();
        ctx.arc(x, y, 5, 0, 2 * Math.PI);
        ctx.fill();
        ctx.fillText(points.length, x + 7, y - 7);
    }};

    function sendPoints() {{
        google.colab.kernel.invokeFunction('notebook.get_points', [points], {{}});
    }}
</script>
"""

import base64
_, buffer = cv2.imencode('.jpg', img)
img_base64 = base64.b64encode(buffer).decode('utf-8')

puntos_seleccionados = []

def get_points(pts):
    global puntos_seleccionados
    puntos_seleccionados = pts
    print(f"✅ Puntos capturados: {puntos_seleccionados}")

output.register_callback('notebook.get_points', get_points)

display(HTML(canvas_html.format(w=ancho, h=alto, data=img_base64)))

In [ ]:
# @title Definición de Medidas Reales
# @markdown Introduce las medidas en metros del rectángulo que marcaste:
largo_metros = 2 # @param {type:"number"}
ancho_metros = 4.5 # @param {type:"number"}

if len(puntos_seleccionados) == 4:
    # Puntos en el video (píxeles)
    pts_src = np.array(puntos_seleccionados, dtype='float32')

    # Puntos en el mapa real (metros)
    # Definimos un rectángulo que empieza en (0,0)
    pts_dst = np.array([
        [0, 0],              # Sup-Izq
        [ancho_metros, 0],   # Sup-Der
        [ancho_metros, largo_metros], # Inf-Der
        [0, largo_metros]    # Inf-Izq
    ], dtype='float32')

    # Calculamos la matriz de homografía
    matriz_h, _ = cv2.findHomography(pts_src, pts_dst)

    # Guardamos la matriz para no tener que repetir esto
    # Corrected path for saving homography matrix
    np.save('/content/drive/MyDrive/Analisis_Deportivo/Configuracion/matriz_h.npy', matriz_h)
    print("✅ Calibración espacial completada y guardada.")
else:
    print("❌ Error: Debes seleccionar exactamente 4 puntos en el paso anterior.")

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import cv2
import numpy as np
from sklearn.cluster import KMeans
from google.colab.patches import cv2_imshow

# Cargamos un modelo pequeño y rápido
model = YOLO('yolov8n.pt')

In [ ]:
def es_verde(color_hsv):
    """Detecta si un color está en el rango del verde (pasto)"""
    # Rangos típicos de verde en HSV
    bajo_verde = np.array([35, 40, 40])
    alto_verde = np.array([85, 255, 255])
    return cv2.inRange(np.array([[color_hsv]], dtype=np.uint8), bajo_verde, alto_verde)[0][0] > 0

def obtener_color_camiseta(crop):
    # Convertimos a HSV
    hsv_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    # Solo tomamos la parte central/superior para evitar fondos
    alto, ancho, _ = hsv_crop.shape
    roi = hsv_crop[int(alto*0.2):int(alto*0.5), int(ancho*0.3):int(ancho*0.7)]

    # Aplanamos los píxeles y filtramos los verdes
    pixeles = roi.reshape(-1, 3)
    pixeles_sin_verde = [p for p in pixeles if not es_verde(p)]

    if len(pixeles_sin_verde) < 10: return None

    # Usamos K-Means para encontrar el color más representativo
    kmeans = KMeans(n_clusters=1, n_init=10)
    kmeans.fit(pixeles_sin_verde)
    return kmeans.cluster_centers_[0]

# 1. Detectar jugadores en el frame
ruta_frame = '/content/drive/MyDrive/Analisis_Deportivo/Configuracion/frame_calibracion.jpg' # Corrected path
frame = cv2.imread(ruta_frame)
results = model(frame, classes=[0], conf=0.5) # Clase 0 es 'persona'

muestras_colores = []
print("Extrayendo muestras de jugadores...")

for r in results:
    boxes = r.boxes.xyxy.cpu().numpy()
    for i, box in enumerate(boxes[:10]): # Tomamos máximo 10 muestras
        x1, y1, x2, y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]

        color = obtener_color_camiseta(crop)
        if color is not None:
            muestras_colores.append(color)
            # Guardamos miniatura para que el usuario identifique
            cv2.imwrite(f'muestra_{i}.jpg', crop)

print(f"✅ Se obtuvieron {len(muestras_colores)} muestras de color.")

In [ ]:
import json

# Definición de firmas de color en HSV
# Verde Local: Un verde generalmente más oscuro o saturado que el pasto
color_verde_local = [60, 180, 150]

# Azul Visitante: Un azul estándar
color_azul_visita = [120, 200, 150]

ruta_json = '/content/drive/MyDrive/Analisis_Futbol/Configuracion/colores_partido.json'

config_color = {
    "local": color_verde_local,
    "visita": color_azul_visita,
    "nombres": {"local": "Local (Verde)", "visita": "Visitante (Azul)"}
}

with open(ruta_json, 'w') as f:
    json.dump(config_color, f)

print("✅ Configuración actualizada: Local = VERDE, Visitante = AZUL")

In [ ]:
# @title Asignación de Equipos
# @markdown Mira las imágenes generadas (muestra_0.jpg, muestra_1.jpg, etc.) en el explorador de archivos a la izquierda.
indice_local = 0 # @param {type:"integer"}
indice_visita = 1 # @param {type:"integer"}

color_local = muestras_colores[indice_local]
color_visita = muestras_colores[indice_visita]

# Guardamos los colores "Firma" del partido
config_color = {
    "local": color_local.tolist(),
    "visita": color_visita.tolist()
}

import json
# Corrected path for saving colors
with open('/content/drive/MyDrive/Analisis_Deportivo/Configuracion/colores_partido.json', 'w') as f:
    json.dump(config_color, f)

print("✅ ADN de equipos guardado.")
print(f"Color Local (HSV): {color_local}")
print(f"Color Visita (HSV): {color_visita}")

In [ ]:
import cv2
import numpy as np
import json
import sqlite3
import os
from ultralytics import YOLO

# 1. CARGA DE CONFIGURACIÓN Y MODELOS
ruta_raiz = '/content/drive/MyDrive/Analisis_Deportivo' # Corrected path
matriz_h = np.load(os.path.join(ruta_raiz, 'Configuracion', 'matriz_h.npy'))
with open(os.path.join(ruta_raiz, 'Configuracion', 'colores_partido.json'), 'r') as f:
    config_color = json.load(f)

model = YOLO('yolov8n.pt') # Modelo base para detección
color_objetivo = np.array(config_color['local']) # El verde del arquero

# 2. CONEXIÓN A BASE DE DATOS
db_path = os.path.join(ruta_raiz, 'Resultados', 'analisis_arquero.db') # Corrected path
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS datos_arquero (
        frame INTEGER,
        x_real REAL,
        y_real REAL,
        distancia_arco REAL,
        velocidad REAL
    )
''')

# 3. PROCESAMIENTO DE VIDEO
video_input = '/content/drive/MyDrive/Hue vsArs - 10 mayo 26.mp4' # Corrected path
cap = cv2.VideoCapture(video_input)
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Preparar video de salida para revisión
video_output = os.path.join(ruta_raiz, 'Resultados', 'arquero_analizado.mp4') # Corrected path
out = cv2.VideoWriter(video_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

frame_count = 0
ultima_pos = None

print("Iniciando análisis del arquero...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Detección de personas
    results = model.predict(frame, classes=[0], conf=0.4, verbose=False)

    arquero_detectado = None
    min_dist_color = float('inf')

    for r in results:
        for box in r.boxes.xyxy:
            x1, y1, x2, y2 = map(int, box)

            # Analizar color del pecho/torso
            recorte = frame[y1:int(y1+(y2-y1)*0.5), x1:x2]
            if recorte.size == 0: continue

            hsv_recorte = cv2.cvtColor(recorte, cv2.COLOR_BGR2HSV)
            color_promedio = cv2.mean(hsv_recorte)[:3]

            # Calcular qué tan cerca está del "Verde" configurado
            distancia = np.linalg.norm(np.array(color_promedio) - color_objetivo)

            if distancia < min_dist_color:
                min_dist_color = distancia
                arquero_detectado = (x1, y1, x2, y2)

    # Si encontramos al arquero, procesamos sus datos
    if arquero_detectado:
        x1, y1, x2, y2 = arquero_detectado
        # Punto base (pies del arquero) para la homografía
        punto_pies = np.array([[(x1 + x2) / 2, y2]], dtype='float32')

        # Transformación a metros reales
        punto_real = cv2.perspectiveTransform(np.array([punto_pies]), matriz_h)[0][0]
        x_r, y_r = punto_real[0], punto_real[1]

        # Variable: Distancia al arco (asumiendo arco en x=0, y=ancho/2)
        # Ajustar estas coordenadas según tu calibración
        dist_arco = np.sqrt((x_r - 0)**2 + (y_r - 20.16)**2)

        # Variable: Velocidad (Diferencia de posición / tiempo)
        velocidad = 0
        if ultima_pos is not None:
            dist_recorrida = np.sqrt((x_r - ultima_pos[0])**2 + (y_r - ultima_pos[1])**2)
            velocidad = dist_recorrida * fps # metros por segundo

        ultima_pos = (x_r, y_r)

        # Guardar en Base de Datos
        cursor.execute('INSERT INTO datos_arquero VALUES (?, ?, ?, ?, ?)',
                       (frame_count, float(x_r), float(y_r), float(dist_arco), float(velocidad)))

        # Dibujar en el video para validación
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"Arquero: {dist_arco:.1f}m al arco", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    out.write(frame)
    frame_count += 1
    if frame_count % 100 == 0: print(f"Procesados {frame_count} cuadros...")

conn.commit()
cap.release()
out.release()
conn.close()
print(f"✅ Análisis completo. Datos guardados en SQLite y video generado en la carpeta Resultados.")

In [ ]:
import pandas as pd
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM datos_arquero", conn)
print(df.describe()) # Te dará promedios de distancia, velocidad, etc.

In [ ]:
# Conexión y actualización de tabla
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# Drop the table if it exists to ensure the schema is always up-to-date
cursor.execute('DROP TABLE IF EXISTS analisis_avanzado_arquero')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS analisis_avanzado_arquero (
        frame INTEGER,
        ark_x REAL, ark_y REAL,
        ball_x REAL, ball_y REAL,
        distancia_balon_arco REAL,
        evento TEXT, -- 'remate', 'atajada', 'posicionamiento'
        tiempo_reaccion_ms REAL,
        cobertura_pct REAL
    )
''')

In [ ]:
import cv2
import numpy as np
import json
import sqlite3
import os
from ultralytics import YOLO

# Reload necessary data for this cell's execution, in case kernel restarted or dependencies changed
ruta_raiz = '/content/drive/MyDrive/Analisis_Deportivo' # Corrected path
matriz_h = np.load(os.path.join(ruta_raiz, 'Configuracion', 'matriz_h.npy'))
with open(os.path.join(ruta_raiz, 'Configuracion', 'colores_partido.json'), 'r') as f:
    config_color = json.load(f)

model = YOLO('yolov8n.pt') # Modelo base para detección
color_objetivo = np.array(config_color['local']) # The color of the local team, assumed to be the goalkeeper's

# Connect to database (assuming db_path is defined globally or re-defined here)
db_path = os.path.join(ruta_raiz, 'Resultados', 'analisis_arquero.db') # Assuming it's the same db for now, or could be a new one if specified for advanced analysis
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Ensure analisis_avanzado_arquero table exists for this connection
cursor.execute('DROP TABLE IF EXISTS analisis_avanzado_arquero') # Drop to ensure latest schema
cursor.execute('''
    CREATE TABLE IF NOT EXISTS analisis_avanzado_arquero (
        frame INTEGER,
        ark_x REAL, ark_y REAL,
        ball_x REAL, ball_y REAL,
        distancia_balon_arco REAL,
        evento TEXT, -- 'remate', 'atajada', 'posicionamiento'
        tiempo_reaccion_ms REAL,
        cobertura_pct REAL
    )
''')

print("Iniciando análisis integral: Arquero + Balón...")

# Variables de estado para lógica de eventos
en_remate = False
frame_inicio_remate = 0

video_input = '/content/drive/MyDrive/Hue vsArs - 10 mayo 26.mp4' # Corrected path
cap = cv2.VideoCapture(video_input)
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Preparar video de salida para revisión
video_output_integral = os.path.join(ruta_raiz, 'Resultados', 'analisis_integral.mp4') # Corrected path
out = cv2.VideoWriter(video_output_integral, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

frame_count = 0
ultima_pos = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Detección de personas (0) y balones (32)
    results = model.predict(frame, classes=[0, 32], conf=0.3, verbose=False)

    pos_ark_real = None
    pos_ball_real = None

    # Lógica para identificar al arquero por color (reutilizamos la anterior)
    arquero_detectado = None
    min_dist_color = float('inf')

    for r in results:
        for box_obj in r.boxes:
            # Check for person (class 0) to find goalkeeper
            if int(box_obj.cls) == 0:
                x1, y1, x2, y2 = map(int, box_obj.xyxy[0])
                recorte = frame[y1:int(y1+(y2-y1)*0.5), x1:x2]
                if recorte.size > 0:
                    hsv_recorte = cv2.cvtColor(recorte, cv2.COLOR_BGR2HSV)
                    color_promedio = cv2.mean(hsv_recorte)[:3]
                    distancia = np.linalg.norm(np.array(color_promedio) - color_objetivo)
                    if distancia < min_dist_color:
                        min_dist_color = distancia
                        arquero_detectado = (x1, y1, x2, y2)

            # Lógica para el balón (class 32)
            elif int(box_obj.cls) == 32: # Es el balón
                bx1, by1, bx2, by2 = map(int, box_obj.xyxy[0])
                punto_balon = np.array([[(bx1 + bx2) / 2, by2]], dtype='float32')
                pos_ball_real = cv2.perspectiveTransform(np.array([punto_balon]), matriz_h)[0][0]

    # If goalkeeper is detected, calculate its real position
    if arquero_detectado:
        x1_ark, y1_ark, x2_ark, y2_ark = arquero_detectado
        punto_pies_ark = np.array([[(x1_ark + x2_ark) / 2, y2_ark]], dtype='float32')
        pos_ark_real = cv2.perspectiveTransform(np.array([punto_pies_ark]), matriz_h)[0][0]
        # Draw bounding box for goalkeeper
        cv2.rectangle(frame, (x1_ark, y1_ark), (x2_ark, y2_ark), (0, 255, 0), 2) # Green for goalkeeper

    # Initialize values for database insertion
    dist_balon_arco = None
    evento_actual = "posicionamiento"
    reaccion = 0
    porcentaje_cobertura = None # Initialize new variable

    if pos_ball_real is not None and pos_ark_real is not None:
        # Calcular distancia del balón al arco (punto central del arco x=0, y=20.16)
        dist_balon_arco = np.sqrt((pos_ball_real[0] - 0)**2 + (pos_ball_real[1] - 20.16)**2)

        # INTEGRATED CODE FOR ANGULAR COVERAGE
        ang_total, ang_cubierto = calcular_angulo_cobertura(pos_ball_real, pos_ark_real)
        porcentaje_cobertura = (ang_cubierto / ang_total) * 100 if ang_total > 0 else 0

        # LÓGICA DE EVENTOS
        # ¿Es un remate? (Lógica simple: si el balón está cerca y se mueve hacia el arco)
        if dist_balon_arco < 25 and not en_remate:
            en_remate = True
            frame_inicio_remate = frame_count
            evento_actual = "remate"
            print(f"¡Remate detectado en frame {frame_count}!")

        # ¿Reacción del arquero?
        if en_remate:
            # Si el arquero se movió más de 0.5m desde el inicio del remate
            # Calculamos el tiempo transcurrido en ms
            reaccion = (frame_count - frame_inicio_remate) * (1000 / fps)

            # Si el balón se aleja o se detiene, cerramos el evento
            if dist_balon_arco < 1: # Atajada o Gol
                evento_actual = "atajada"
                en_remate = False

    # Guardar en Base de Datos
    cursor.execute('INSERT INTO analisis_avanzado_arquero VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
                   (frame_count,
                    float(pos_ark_real[0]) if pos_ark_real is not None else None,
                    float(pos_ark_real[1]) if pos_ark_real is not None else None,
                    float(pos_ball_real[0]) if pos_ball_real is not None else None,
                    float(pos_ball_real[1]) if pos_ball_real is not None else None,
                    float(dist_balon_arco) if dist_balon_arco is not None else None,
                    evento_actual,
                    float(reaccion),
                    float(porcentaje_cobertura) if porcentaje_cobertura is not None else None))

    # ... (Dibujar en video y guardar frame)
    out.write(frame)
    frame_count += 1
    if frame_count % 100 == 0: print(f"Procesados {frame_count} cuadros...")

conn.commit()
cap.release()
out.release()
conn.close()
print(f"✅ Análisis completo. Datos guardados en SQLite y video generado en la carpeta Resultados.")

In [ ]:
import pandas as pd
df = pd.read_sql_query("SELECT * FROM analisis_avanzado_arquero WHERE evento = 'remate'", conn)
print(f"Distancia promedio de disparos recibidos: {df['distancia_balon_arco'].mean():.2f} metros")

In [ ]:
import math

def calcular_angulo_cobertura(pos_ball, pos_ark):
    # Coordenadas de los postes (en metros reales)
    p1 = np.array([0, 16.50])
    p2 = np.array([0, 23.82])

    # Vectores desde el balón a los postes
    v_ball_p1 = p1 - pos_ball
    v_ball_p2 = p2 - pos_ball

    # Ángulo total disponible para el rematador (en grados)
    dot_p = np.dot(v_ball_p1, v_ball_p2)
    norm_p = np.linalg.norm(v_ball_p1) * np.linalg.norm(v_ball_p2)
    angulo_total = math.degrees(math.acos(clip(dot_p / norm_p, -1.0, 1.0)))

    # Vector desde el balón al arquero
    v_ball_ark = pos_ark - pos_ball

    # Ángulo que ocupa el arquero (estimando su ancho efectivo de 1.5 metros)
    dist_ball_ark = np.linalg.norm(v_ball_ark)
    if dist_ball_ark > 0:
        # Aproximación del ángulo que cubre el cuerpo del arquero
        angulo_ark = math.degrees(2 * math.atan(0.75 / dist_ball_ark))
    else:
        angulo_ark = 0

    return angulo_total, angulo_ark

# Función auxiliar para evitar errores de precisión numérica
def clip(n, minn, maxn):
    return max(min(n, maxn), minn)

In [ ]:
if pos_ball_real is not None and pos_ark_real is not None:
    ang_total, ang_cubierto = calcular_angulo_cobertura(pos_ball_real, pos_ark_real)

    # Porcentaje de arco cubierto
    porcentaje_cobertura = (ang_cubierto / ang_total) * 100

    # Guardar en SQLite (asegúrate de haber añadido la columna 'cobertura_pct')
    cursor.execute('UPDATE analisis_avanzado_arquero SET cobertura_pct = ? WHERE frame = ?',
                   (float(porcentaje_cobertura), frame_count))